# Phase 1 - Database Creation

#### Phase 1 Goal - Extract job postings from LinkedIn job alerts in a GMail inbox, clean/standardize them, and store them in a database.

## Job Assistant — Phase 1 Overview

### Goal:
##### Build a Python-based system that collects job postings from
##### LinkedIn job-alert emails and stores them in a
##### structured local database.

### Phase 1:
##### Gmail → Python → Job Parser → SQLite

### Later phases:
##### Phase 2 → Resume/job matching
##### Phase 3 → Tailored resume generation
##### Phase 4 → Email report + PDF attachments
##### Phase 5 → Automation / production deployment

In [104]:
%pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [105]:
# ============================================
# PART 1 - IMPORT LIBRARIES
# ============================================

# Standard library
import base64
import re
import sqlite3
import sys
from datetime import datetime, timezone
from pathlib import Path

# Third-party libraries
import pandas as pd
from bs4 import BeautifulSoup

# Google Gmail API
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Google OAuth authentication
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:11:29) [Clang 20.1.8 ]
Python executable: /opt/anaconda3/bin/python
Working directory: /Users/rushikpatel/Job_Assistant


In [106]:
# ============================================
# PART 2 — PROJECT CONFIGURATION
# ============================================
# --------------------------------------------
# 1. Define the main project folder.
# --------------------------------------------

PROJECT_DIR = Path.cwd()

# --------------------------------------------
# 2. Define important files and folders.
# --------------------------------------------

DATA_DIR = PROJECT_DIR / "data"

CREDENTIALS_FILE = PROJECT_DIR / "credentials.json"
TOKEN_FILE = PROJECT_DIR / "token.json"

DATABASE_FILE = DATA_DIR / "jobs.db"
CSV_FILE = DATA_DIR / "jobs_review.csv"

# --------------------------------------------
# 3. Create the data folder if it doesn't exist.
# --------------------------------------------

DATA_DIR.mkdir(exist_ok=True)

# --------------------------------------------
# 4. Configure Gmail API for Gmail search.
# --------------------------------------------

GMAIL_SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly"
]

# --------------------------------------------
# 5. Gmail search query to be used during job search.
# --------------------------------------------

GMAIL_QUERY = 'newer_than:7d (LinkedIn)'

# --------------------------------------------
# 6. Display the configuration to ensure correct setup.
# --------------------------------------------

print("Project directory:")
print(PROJECT_DIR)

print("\nData directory:")
print(DATA_DIR)

print("\nCredentials file:")
print(CREDENTIALS_FILE)

print("\nToken file:")
print(TOKEN_FILE)

print("\nDatabase file:")
print(DATABASE_FILE)

print("\nCSV export:")
print(CSV_FILE)

print("\nGmail query:")
print(GMAIL_QUERY)

Project directory:
/Users/rushikpatel/Job_Assistant

Data directory:
/Users/rushikpatel/Job_Assistant/data

Credentials file:
/Users/rushikpatel/Job_Assistant/credentials.json

Token file:
/Users/rushikpatel/Job_Assistant/token.json

Database file:
/Users/rushikpatel/Job_Assistant/data/jobs.db

CSV export:
/Users/rushikpatel/Job_Assistant/data/jobs_review.csv

Gmail query:
newer_than:7d (LinkedIn)


In [107]:
# ============================================
# PART 3 — GMAIL API AUTHENTICATION
# ============================================

def authenticate_gmail():
    """
    Authenticate with the Gmail API using OAuth 2.0.

    The first time this function runs, Google will open
    a browser window asking for access authorization.

    After successful authorization, a token.json file
    will be created so you don't have to authorize
    every time.
    """

    creds = None

    # --------------------------------------------
    # 1. Check whether we already have a token.
    # --------------------------------------------

    if TOKEN_FILE.exists():
        creds = Credentials.from_authorized_user_file(
            TOKEN_FILE,
            GMAIL_SCOPES
        )

    # --------------------------------------------
    # 2. Check whether the credentials are valid.
    # --------------------------------------------

    if not creds or not creds.valid:

        # If the credentials expired but can be refreshed,
        # refresh them instead of asking the user to log in again.
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())

        # Otherwise, start the OAuth authorization process.
        else:

            if not CREDENTIALS_FILE.exists():
                raise FileNotFoundError(
                    f"Could not find {CREDENTIALS_FILE}.\n"
                    "Make sure credentials.json is in your project folder."
                )

            flow = InstalledAppFlow.from_client_secrets_file(
                CREDENTIALS_FILE,
                GMAIL_SCOPES
            )

            creds = flow.run_local_server(port=0)

        # ----------------------------------------
        # 3. Save the credentials for future runs.
        # ----------------------------------------

        with open(TOKEN_FILE, "w") as token:
            token.write(creds.to_json())

    # --------------------------------------------
    # 4. Build the Gmail API service.
    # --------------------------------------------

    service = build(
        "gmail",
        "v1",
        credentials=creds
    )

    return service


# Run the authentication function.
gmail_service = authenticate_gmail()

print("Gmail authentication successful!")

Gmail authentication successful!


In [108]:
# ============================================
# PART 4 — SEARCH GMAIL FOR LINKEDIN ALERTS
# ============================================

def search_gmail(query, max_results=50):
    
    try:
        response = gmail_service.users().messages().list(
            userId="me",
            q=query,
            maxResults=max_results
        ).execute()

        messages = response.get("messages", [])

        return messages

    except HttpError as error:
        print(f"Gmail API error: {error}")
        return []


# Search for recent LinkedIn emails using previously-defined Gmail search query.
messages = search_gmail(
    GMAIL_QUERY,
    max_results=50
)

print(f"Found {len(messages)} matching LinkedIn emails.")

Found 12 matching LinkedIn emails.


In [109]:
# ============================================
# PART 5 — RETRIEVE A GMAIL MESSAGE
# ============================================

# Retrieve the full Gmail message using its message ID.

def get_gmail_message(message_id):

    try:
        message = gmail_service.users().messages().get(
            userId="me",
            id=message_id,
            format="full"
        ).execute()

        return message

    except HttpError as error:
        print(f"Gmail API error: {error}")
        return None

# Retrieve the first matching LinkedIn email

if messages:
    first_message = get_gmail_message(messages[0]["id"])

    print("Successfully retrieved email.")
    print(f"Message ID: {first_message['id']}")
    print(f"Thread ID: {first_message['threadId']}")
    
else:
    print("No LinkedIn emails were found.")

Successfully retrieved email.
Message ID: 1a06dd8d0f6a4888
Thread ID: 1a06dd8d0f6a4888


In [110]:
# ============================================
# PART 6 — EXTRACT EMAIL BODY
# ============================================

# Decode a Gmail email body from Base64 URL-safe encoding.

def decode_email_body(data):
 
    if not data:
        return ""

    decoded_bytes = base64.urlsafe_b64decode(data + "===")

    return decoded_bytes.decode(
        "utf-8",
        errors="replace"
    )

def extract_email_body(payload):
    """
    Recursively extract text and HTML content
    from a Gmail message payload.
    """

    text_body = ""
    html_body = ""

    # --------------------------------------------
    # Check the current part.
    # --------------------------------------------

    mime_type = payload.get("mimeType", "")
    body = payload.get("body", {})
    data = body.get("data")

    if data:

        decoded = decode_email_body(data)

        if mime_type == "text/plain":
            text_body += decoded

        elif mime_type == "text/html":
            html_body += decoded

    # --------------------------------------------
    # Check nested parts.
    # --------------------------------------------

    for part in payload.get("parts", []):
        part_text, part_html = extract_email_body(part)

        text_body += part_text
        html_body += part_html

    return text_body, html_body


# Extract the body from our first LinkedIn email.
text_body, html_body = extract_email_body(
    first_message["payload"]
)

print("Plain-text characters:", len(text_body))
print("HTML characters:", len(html_body))

Plain-text characters: 2182
HTML characters: 90988


In [111]:
# ============================================
# PART 7 — LINKEDIN JOB PARSING
# ============================================

# 7A - Create job records with standardized format.

def create_job_record(
    source="",
    company="",
    title="",
    location="",
    description="",
    url="",
    salary="",
    remote_status="",
    date_posted="",
    date_found="",
    job_id=""
):
    return {
        "source": source,
        "company": company,
        "title": title,
        "location": location,
        "description": description,
        "url": url,
        "salary": salary,
        "remote_status": remote_status,
        "date_posted": date_posted,
        "date_found": date_found,
        "job_id": job_id
    }

In [112]:
# 7B - Normalize job URLs into clean formatting containing only the LinkedIn job ID.

def normalize_linkedin_job_url(url):
  
    match = re.search(
        r"linkedin\.com/comm/jobs/view/(\d+)",
        url
    )

    if match:
        job_id = match.group(1)
        return f"https://www.linkedin.com/comm/jobs/view/{job_id}/"

    return url

In [113]:
# 7C - Extract the numeric LinkedIN job ID from a job URL, to log/track individual job records in the database.

def extract_linkedin_job_id(url):

    match = re.search(
        r"linkedin\.com/comm/jobs/view/(\d+)",
        url
    )

    if match:
        return match.group(1)

    return None

In [114]:
# 7D - Extract relevant information from a LinkedIn job-alert email and generate a list of unique job dictionaries.

def parse_linkedin_job_email(html):
    
    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")

    jobs = []
    seen_job_ids = set()

    for link in soup.find_all("a", href=True):

        href = link["href"]

        # Identify LinkedIn job ID
        match = re.search(
            r"linkedin\.com/comm/jobs/view/(\d+)",
            href
        )

        if not match:
            continue

        job_id = match.group(1)

        # Prevent duplicate jobs
        if job_id in seen_job_ids:
            continue

        # Get the text associated with this link
        link_text = link.get_text(" ", strip=True)

        # Skip links that don't contain a meaningful title
        if not link_text:
            continue

        # Skip generic LinkedIn buttons
        if link_text.lower() in ["view job", "apply"]:
            continue

        # Look for company/location information
        company = ""
        location = ""

        parent_td = link.find_parent("td")

        if parent_td:

            location_element = parent_td.find_next("p")

            if location_element:

                company_location = location_element.get_text(
                    " ",
                    strip=True
                )

                if "·" in company_location:

                    parts = company_location.split("·", 1)

                    company = parts[0].strip()
                    location = parts[1].strip()

        # Clean title
        title = link_text

        # Remove common status text if present
        title = re.sub(
            r"\s+Actively recruiting$",
            "",
            title,
            flags=re.IGNORECASE
        ).strip()

        # If the link contains company/location text,
        # remove it from the title.
        if company and location:

            unwanted_text = f"{company} · {location}"

            title = title.replace(
                unwanted_text,
                ""
            ).strip()

        # Create job record
               # Create job record
        job = create_job_record(
            source="LinkedIn",
            company=company,
            title=title,
            location=location,
            url=href,
            date_found=datetime.now(timezone.utc).isoformat(),
            job_id=job_id
        )

        jobs.append(job)
        seen_job_ids.add(job_id)

    return jobs

In [115]:
# ============================================
# PART 8 — GMAIL MESSAGE PROCESSING
# ============================================

def process_gmail_message(message_id):
    """
    Retrieve one Gmail message and extract LinkedIn jobs from it.

    Parameters:
        message_id: Gmail message ID

    Returns:
        List of job dictionaries
    """

    # Retrieve the full Gmail message
    message = get_gmail_message(message_id)

    if not message:
        return []

    # Extract the plain-text and HTML versions
    text_body, html_body = extract_email_body(
        message["payload"]
    )

    # Parse LinkedIn jobs from the HTML
    jobs = parse_linkedin_job_email(
        html_body
    )

    # Clean the LinkedIn URLs
    for job in jobs:
        job["url"] = normalize_linkedin_job_url(
            job["url"]
        )

    return jobs

In [116]:
# ============================================
# PART 9 - REMOVE DUPLICATE JOB RECORDS
# ============================================

def deduplicate_jobs(jobs):
    """
    Remove duplicate LinkedIn jobs using the LinkedIn job ID.
    """

    unique_jobs = {}

    for job in jobs:

        # Extract the LinkedIn job ID from the URL
        match = re.search(
            r"linkedin\.com/comm/jobs/view/(\d+)",
            job["url"]
        )

        if match:
            job_id = match.group(1)

            # Only keep the first occurrence
            if job_id not in unique_jobs:
                unique_jobs[job_id] = job

        else:
            # Keep records without a recognizable job ID
            unique_jobs[job["url"]] = job

    return list(unique_jobs.values())

In [117]:
# ============================================
# PART 10 — DATABASE MANAGEMENT
# ============================================

# 10A - Create SQLite database to store job records.

def create_jobs_table():
    """
    Create the jobs table if it does not already exist.
    """

    connection = sqlite3.connect(DATABASE_FILE)

    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS jobs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source TEXT NOT NULL,
            company TEXT NOT NULL,
            title TEXT NOT NULL,
            location TEXT NOT NULL,
            description TEXT,
            url TEXT NOT NULL UNIQUE,
            salary TEXT,
            remote_status TEXT,
            date_posted TEXT,
            date_found TEXT NOT NULL
        )
    """)

    connection.commit()
    connection.close()

    print("Jobs table created successfully.")


create_jobs_table()

Jobs table created successfully.


In [118]:
# 10B - Insert collected job records into the SQLite database, ignoring existing jobs to avoid duplicates.

def insert_jobs(jobs):
   
    connection = sqlite3.connect(DATABASE_FILE)

    cursor = connection.cursor()

    inserted_count = 0
    skipped_count = 0

    for job in jobs:

        try:
            # Make sure the job has a LinkedIn job ID
            job_id = job.get("job_id")

            if not job_id:
                job_id = extract_linkedin_job_id(
                    job["url"]
                )

            cursor.execute("""
                INSERT INTO jobs (
                    source,
                    company,
                    title,
                    location,
                    description,
                    url,
                    salary,
                    remote_status,
                    date_posted,
                    date_found,
                    job_id
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                job["source"],
                job["company"],
                job["title"],
                job["location"],
                job["description"],
                job["url"],
                job["salary"],
                job["remote_status"],
                job["date_posted"],
                job["date_found"],
                job_id
            ))

            inserted_count += 1

        except sqlite3.IntegrityError:
            skipped_count += 1

    connection.commit()
    connection.close()

    print(f"Jobs inserted: {inserted_count}")
    print(f"Jobs skipped (already in database): {skipped_count}")
insert_jobs(unique_jobs)

Jobs inserted: 0
Jobs skipped (already in database): 34


In [119]:
# 10C - Retrieve all job records currently stored in the database.

def get_all_jobs_from_database():
   
    connection = sqlite3.connect(DATABASE_FILE)

    query = """
        SELECT
            id,
            job_id,
            source,
            company,
            title,
            location,
            description,
            url,
            salary,
            remote_status,
            date_posted,
            date_found
        FROM jobs
        ORDER BY id
    """

    df = pd.read_sql_query(
        query,
        connection
    )

    connection.close()

    return df
    
database_jobs_df = get_all_jobs_from_database()

print(f"Jobs currently in database: {len(database_jobs_df)}")

database_jobs_df

Jobs currently in database: 34


,id,job_id,source,company,title,location,description,url,salary,remote_status,date_posted,date_found
0,1,4459997344,LinkedIn,UL Solutions,Engineering Leader - Water Systems,"Northbrook, IL",,https://www.linkedin.com/comm/jobs/view/445999...,,,,2026-09-04T13:54:49.259531+00:00
1,2,4461356107,LinkedIn,IQVIA,"Clinical Project Manager, Internal Medicine, I...","Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446135...,,,,2026-09-04T13:54:49.800636+00:00
2,3,4459432027,LinkedIn,AbbVie,Associate Business Systems Consultant – Medica...,"Mettawa, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/445943...,,,,2026-09-04T13:54:49.800761+00:00
3,4,4461345415,LinkedIn,AbbVie,"Director, Global Program Manager - Immunology","North Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446134...,,,,2026-09-04T13:54:49.800855+00:00
4,5,4460568387,LinkedIn,UChicago Medicine,Sr Quality Project Manager 2 company alumni,"Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446056...,,,,2026-09-04T13:54:49.800940+00:00
5,6,4459916121,LinkedIn,Abbott,R&D- Project Manager,"Lake Forest, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/445991...,,,,2026-09-04T13:54:49.801023+00:00
6,7,4461553312,LinkedIn,UChicago Medicine,"Project Manager, Care Experience & Culture Tra...","Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446155...,,,,2026-09-04T13:54:49.801110+00:00
7,8,4460326554,LinkedIn,Amazon,Operations Manager 1 connection,"Skokie, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446032...,,,,2026-09-04T13:54:49.990059+00:00
8,9,4458819092,LinkedIn,Charles Schwab,"Manager, Field Operations 2 company alumni","Chicago, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/445881...,,,,2026-09-04T13:54:49.990151+00:00
9,10,4460801874,LinkedIn,Amazon,Operations Manager 23 company alumni,"Romeoville, Illinois, United States",,https://www.linkedin.com/comm/jobs/view/446080...,,,,2026-09-04T13:54:49.990225+00:00


In [120]:
# ============================================
# PART 11 - EXPORT DATABASE TO CSV
# ============================================

# CSV file allows for streamlined human review of collected job records.

database_jobs_df.to_csv(
    CSV_FILE,
    index=False
)

print(f"CSV file created successfully:")
print(CSV_FILE)

CSV file created successfully:
/Users/rushikpatel/Job_Assistant/data/jobs_review.csv


In [121]:
csv_check_df = pd.read_csv(CSV_FILE)

print(f"Rows in CSV: {len(csv_check_df)}")
print(f"Columns in CSV: {len(csv_check_df.columns)}")

csv_check_df.head()

Rows in CSV: 34
Columns in CSV: 12


,id,job_id,source,company,title,location,description,url,salary,remote_status,date_posted,date_found
0,1,4459997344,LinkedIn,UL Solutions,Engineering Leader - Water Systems,"Northbrook, IL",NaN,https://www.linkedin.com/comm/jobs/view/445999...,NaN,NaN,NaN,2026-09-04T13:54:49.259531+00:00
1,2,4461356107,LinkedIn,IQVIA,"Clinical Project Manager, Internal Medicine, I...","Chicago, Illinois, United States",NaN,https://www.linkedin.com/comm/jobs/view/446135...,NaN,NaN,NaN,2026-09-04T13:54:49.800636+00:00
2,3,4459432027,LinkedIn,AbbVie,Associate Business Systems Consultant – Medica...,"Mettawa, Illinois, United States",NaN,https://www.linkedin.com/comm/jobs/view/445943...,NaN,NaN,NaN,2026-09-04T13:54:49.800761+00:00
3,4,4461345415,LinkedIn,AbbVie,"Director, Global Program Manager - Immunology","North Chicago, Illinois, United States",NaN,https://www.linkedin.com/comm/jobs/view/446134...,NaN,NaN,NaN,2026-09-04T13:54:49.800855+00:00
4,5,4460568387,LinkedIn,UChicago Medicine,Sr Quality Project Manager 2 company alumni,"Chicago, Illinois, United States",NaN,https://www.linkedin.com/comm/jobs/view/446056...,NaN,NaN,NaN,2026-09-04T13:54:49.800940+00:00


In [122]:
# ============================================
# PART 12 — REPEATABLE JOB RECORD INGESTION FUNCTION
# ============================================

def ingest_linkedin_jobs():
    """
    Search Gmail for LinkedIn job-alert emails,
    extract job postings, deduplicate them,
    and insert them into the database.
    """

    ingestion_start = datetime.now(timezone.utc)

    print("=" * 60)
    print("STARTING LINKEDIN JOB INGESTION")
    print("=" * 60)

    print(f"\nIngestion started: {ingestion_start.isoformat()}")

    # Step 1: Search Gmail
    messages = search_gmail(
        GMAIL_QUERY,
        max_results=50
    )

    print(f"Gmail messages found: {len(messages)}")

    # Step 2: Extract jobs from each message
    all_jobs = []

    for message in messages:
        message_jobs = process_gmail_message(
            message["id"]
        )

        all_jobs.extend(message_jobs)

    print(f"Job records extracted: {len(all_jobs)}")

    # Step 3: Deduplicate jobs
    unique_jobs = deduplicate_jobs(
        all_jobs
    )

    print(f"Unique jobs: {len(unique_jobs)}")

    # Step 4: Insert jobs into database
    insert_jobs(unique_jobs)

    # Step 5: Calculate duration
    ingestion_end = datetime.now(timezone.utc)

    duration = ingestion_end - ingestion_start

    print(f"\nIngestion completed: {ingestion_end.isoformat()}")
    print(f"Ingestion duration: {duration}")

    print("\nIngestion complete.")
    
ingest_linkedin_jobs()

STARTING LINKEDIN JOB INGESTION

Ingestion started: 2026-09-04T20:19:44.722600+00:00
Gmail messages found: 12
Job records extracted: 34
Unique jobs: 34
Jobs inserted: 0
Jobs skipped (already in database): 34

Ingestion completed: 2026-09-04T20:19:47.027455+00:00
Ingestion duration: 0:00:02.304855

Ingestion complete.


In [123]:
# ============================================
# PART 13 — FINAL PHASE 1 VALIDATION
# ============================================

final_check_df = get_all_jobs_from_database()

print("=" * 60)
print("PHASE 1 FINAL DATABASE CHECK")
print("=" * 60)

print(f"\nTotal jobs in database: {len(final_check_df)}")
print(
    f"Unique LinkedIn job IDs: "
    f"{final_check_df['job_id'].nunique()}"
)

print(
    f"Duplicate LinkedIn job IDs: "
    f"{final_check_df['job_id'].duplicated().sum()}"
)

print("\nMissing values:")
print(
    final_check_df[
        [
            "source",
            "company",
            "title",
            "location",
            "url"
        ]
    ].isna().sum()
)

print("\nDuplicate URLs:")
print(
    final_check_df["url"].duplicated().sum()
)

PHASE 1 FINAL DATABASE CHECK

Total jobs in database: 34
Unique LinkedIn job IDs: 34
Duplicate LinkedIn job IDs: 0

Missing values:
source      0
company     0
title       0
location    0
url         0
dtype: int64

Duplicate URLs:
0


# Appendix A

#### Initial database structure did not include a job_id column and it became clear during development that this column would be greatly helpful in logging job records as they are added to the database. The below code was used to add the job_id column to the database and ensure future jobs are automatically assigned a job ID.